In [1]:
import pandas as pd
import numpy as np

from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from xailib.data_loaders.dataframe_loader import prepare_dataframe

from xailib.explainers.lime_explainer import LimeXAITabularExplainer
from xailib.explainers.lore_explainer import LoreTabularExplainer
from xailib.explainers.shap_explainer_tab import ShapXAITabularExplainer

from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

import altair as alt
import pickle

import os

- Riscrivere il codice con due funzioni:
    - una per il preprocessing dei dati
    - una per il plotting
- Check su cosa prendere per le regole e le soglie
- Usare la FI per ordinare le feature  [X]
    - hconcat con la FI (plot accanto a plot)  [X]
- aggiungere il cutoff su entrambi
- inserire progressive disclosure:
    - filtrare per feature a cui è associata Rules
    - filtrare per FI (cutoff)

- Investigare le CR
- Fare la stessa cosa con titanic [X]

- nel paper partire dalla vecchia viz html (linguaggio naturale)

In [2]:
path = os.getcwd()
print(path)

/Users/eleonoracappuccio/xai-visualization_rules_fi/notebooks


In [3]:
source_file = '../datasets/titanic_c.csv'
class_field = 'Survived'
df = pd.read_csv(source_file, skipinitialspace=True, na_values='?', keep_default_na=True)

In [4]:
source_file = '../datasets/german_credit.csv'
class_field = 'default'
# Load and transform dataset
df = pd.read_csv(source_file, skipinitialspace=True, na_values='?', keep_default_na=True)

In [5]:
df, feature_names, class_values, numeric_columns, rdf, real_feature_names, features_map = prepare_dataframe(df, class_field)

In [6]:
inv_dict=[]
for i,el in enumerate(features_map.values()):
    #invert key value in el dict
    d = {v: k for k, v in el.items()}
    for k, v in d.items():
        inv_dict.append(d[k])

### Learning a Random Forest classfier

We train a RF classifier by using the ```sklearn``` library. We start by splitting the dataset into a train and test subsets. 

In [7]:
test_size = 0.3
random_state = 42
X_train, X_test, Y_train, Y_test = train_test_split(df[feature_names], df[class_field],
                                                        test_size=test_size,
                                                        random_state=random_state,
                                                        stratify=df[class_field])


Then we train the model on the training set. 
Once the model has been learned, we use a wrapper class to get access to the model for ```XAI lib```

In [8]:
bb = RandomForestClassifier(n_estimators=20, random_state=random_state)
bb.fit(X_train.values, Y_train.values)
bbox = sklearn_classifier_wrapper(bb)

Select a new instance to be classfied by the model and print the predicted class.

In [9]:
inst = X_train.iloc[128].values
print('Instance ',inst)
print('True class ',Y_train.iloc[128])
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

Instance  [  27 4526    4    2   32    2    2    0    0    0    1    0    1    0
    0    0    0    0    0    0    0    0    0    1    0    0    1    0
    0    0    0    0    1    0    0    0    0    0    0    1    0    0
    1    0    0    1    0    0    0    1    0    1    0    0    0    0
    1    0    1    0    1]
True class  0
Predicted class  [0]


In [10]:
real_inst = inst
real_inst

array([  27, 4526,    4,    2,   32,    2,    2,    0,    0,    0,    1,
          0,    1,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    1,    0,    0,    1,    0,    0,    0,    0,    0,    1,
          0,    0,    0,    0,    0,    0,    1,    0,    0,    1,    0,
          0,    1,    0,    0,    0,    1,    0,    1,    0,    0,    0,
          0,    1,    0,    1,    0,    1])

## Explaining the prediction
We use the explanators of ```XAI lib``` to provide an explantion for the classified instance ```inst```.
Every explainer of ```XAI lib``` takes in input the blackbox to be explained with the corresponding feature names, and a configuration object to initialize the explainer.

### SHAP explainer

In [11]:
explainer = ShapXAITabularExplainer(bbox, feature_names)
config = {'explainer' : 'tree', 'X_train' : X_train.iloc[0:].values}
explainer.fit(config)

In [12]:
exp = explainer.explain(inst)

In [13]:
exp.exp

[array([-9.36117846e-03,  1.12032620e-02, -9.27695620e-03,  2.34854277e-02,
         7.16980504e-03, -6.74858143e-03,  1.62828062e-03,  8.94296419e-03,
         1.05535082e-02,  1.18873255e-03,  5.97444048e-02,  6.01048718e-03,
         4.42259296e-02,  8.23171215e-04, -5.29116292e-05,  3.35190747e-03,
         5.34567691e-03,  1.59816407e-03,  1.32973681e-02, -3.37325406e-04,
        -6.45782153e-03, -1.92963142e-04, -3.43301732e-05, -1.02040224e-02,
        -9.14099568e-05, -1.88173738e-05,  3.73942711e-02,  1.67373617e-02,
        -2.71268261e-04,  4.33195433e-03, -6.42038496e-03,  4.19965582e-03,
        -6.04810835e-02,  2.52984889e-03, -4.70397752e-03,  1.08855784e-03,
         3.60347919e-03,  2.57403676e-03,  2.60612959e-03,  6.01817322e-03,
         1.27007955e-03, -1.29012630e-04, -5.39136185e-04,  2.42969702e-03,
         7.47511264e-03,  2.09428070e-02,  7.15806453e-03,  1.95509790e-02,
        -2.60691783e-02, -6.28495831e-03,  5.63396486e-05,  5.83210604e-03,
         3.4

In [14]:
shap_feature_importance=exp.exp
shap_feature_importance

[array([-9.36117846e-03,  1.12032620e-02, -9.27695620e-03,  2.34854277e-02,
         7.16980504e-03, -6.74858143e-03,  1.62828062e-03,  8.94296419e-03,
         1.05535082e-02,  1.18873255e-03,  5.97444048e-02,  6.01048718e-03,
         4.42259296e-02,  8.23171215e-04, -5.29116292e-05,  3.35190747e-03,
         5.34567691e-03,  1.59816407e-03,  1.32973681e-02, -3.37325406e-04,
        -6.45782153e-03, -1.92963142e-04, -3.43301732e-05, -1.02040224e-02,
        -9.14099568e-05, -1.88173738e-05,  3.73942711e-02,  1.67373617e-02,
        -2.71268261e-04,  4.33195433e-03, -6.42038496e-03,  4.19965582e-03,
        -6.04810835e-02,  2.52984889e-03, -4.70397752e-03,  1.08855784e-03,
         3.60347919e-03,  2.57403676e-03,  2.60612959e-03,  6.01817322e-03,
         1.27007955e-03, -1.29012630e-04, -5.39136185e-04,  2.42969702e-03,
         7.47511264e-03,  2.09428070e-02,  7.15806453e-03,  1.95509790e-02,
        -2.60691783e-02, -6.28495831e-03,  5.63396486e-05,  5.83210604e-03,
         3.4

In [15]:
exp.plot_features_importance()

alt.VConcatChart(...)

## Learning a different model

### Learning a Logistic Regressor

We train a Logistic Regression by using the ```sklearn``` library. We transform the dataset by using a ```Scaler``` to normalize all the attributes.

In [16]:
scaler = preprocessing.StandardScaler().fit(X_train)
X_scaled = scaler.transform(X_train)

bb = LogisticRegression(C=1, penalty='l2')
bb.fit(X_scaled, Y_train.values)
# pass the model to the wrapper to use it in the XAI lib
bbox = sklearn_classifier_wrapper(bb)

In [17]:
# select a record to explain
inst = X_scaled[182]
print('Instance ',inst)
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

Instance  [ 2.27797454  3.35504085  0.94540357  1.07634233  0.04854891 -0.72456474
 -0.43411405  1.65027399 -0.61477862 -0.25898489 -0.80681063  4.17385345
 -0.6435382  -0.32533856 -1.03489416 -0.20412415 -0.22941573 -0.33068147
  1.75885396 -0.34899122 -0.60155441 -0.15294382 -0.09298136 -0.46852129
 -0.12038585 -0.08481889 -0.23623492 -1.21387736 -0.36174054 -0.24943031
  2.15526362 -0.59715086 -0.45485883 -0.73610476 -0.43875307  4.23307441
 -0.65242771 -0.23958675 -0.32533856  0.90192655  4.72581563 -0.2259448
 -3.15238005 -0.54212562 -0.70181003 -0.63024248  2.30354212 -0.40586384
  0.49329429 -0.23958675  2.88675135 -1.59227935 -0.46170508  2.46388049
 -1.33747696 -0.13206764 -0.5        -1.21387736  1.21387736 -0.20412415
  0.20412415]
Predicted class  [1]


In [18]:
X_scaled

array([[-0.7335121 , -0.71300074,  0.0547138 , ..., -0.82380645,
        -0.20412415,  0.20412415],
       [-0.23159766, -0.61086948,  0.0547138 , ..., -0.82380645,
        -0.20412415,  0.20412415],
       [-0.23159766,  0.3606869 , -0.83597597, ..., -0.82380645,
        -0.20412415,  0.20412415],
       ...,
       [-0.23159766, -0.25658997,  0.0547138 , ...,  1.21387736,
        -0.20412415,  0.20412415],
       [-0.23159766,  1.97159246, -1.72666575, ...,  1.21387736,
        -0.20412415,  0.20412415],
       [-0.48255488, -0.06429887, -0.83597597, ..., -0.82380645,
        -0.20412415,  0.20412415]])

## Explaining the prediction
We use the same explainators as for the previous model. In this case, a few adjustments are necessary for the initialization of the explanators. For example, SHAP needs a specific configuration for the linear model we are using.

## LIME tabular explainer

In [19]:
limeExplainer = LimeXAITabularExplainer(bbox)
config = {'feature_selection': 'lasso_path'}
limeExplainer.fit(df, class_field, config)
lime_exp = limeExplainer.explain(inst)
print(lime_exp.exp.as_list())# è una lista di tuple

[('other_debtors=co-applicant', -1.3883374336762273e-09), ('credit_history=all credits at this bank paid back duly', -1.1224119198955936e-09), ('present_emp_since=unemployed', -1.0329114645969508e-09), ('other_debtors=none', 8.17296888233333e-10), ('housing=for free', -5.16434000027017e-10), ('job=management/ self-employed/ highly qualified employee/ officer', -4.078288312147858e-10), ('property=unknown / no property', -3.7867938831658013e-10), ('credit_amount', 3.1576953843122964e-10), ('housing=own', 3.1101205389226697e-10), ('savings=unknown/ no savings account', -2.965422208130157e-10), ('people_under_maintenance', 2.642513724787797e-10), ('foreign_worker=yes', 2.537485978213864e-10), ('credits_this_bank', 2.406761950301443e-10), ('telephone=none', 2.3974592252171386e-10), ('savings=... < 100 DM', 2.2430511967199792e-10), ('purpose=car (new)', -2.1264799070049253e-10), ('job=skilled employee / official', 2.115319882365941e-10), ('credit_history=existing credits paid back duly till 

In [20]:
lime_feature_imp=lime_exp.exp.as_list()
lime_feature_imp

[('other_debtors=co-applicant', -1.3883374336762273e-09),
 ('credit_history=all credits at this bank paid back duly',
  -1.1224119198955936e-09),
 ('present_emp_since=unemployed', -1.0329114645969508e-09),
 ('other_debtors=none', 8.17296888233333e-10),
 ('housing=for free', -5.16434000027017e-10),
 ('job=management/ self-employed/ highly qualified employee/ officer',
  -4.078288312147858e-10),
 ('property=unknown / no property', -3.7867938831658013e-10),
 ('credit_amount', 3.1576953843122964e-10),
 ('housing=own', 3.1101205389226697e-10),
 ('savings=unknown/ no savings account', -2.965422208130157e-10),
 ('people_under_maintenance', 2.642513724787797e-10),
 ('foreign_worker=yes', 2.537485978213864e-10),
 ('credits_this_bank', 2.406761950301443e-10),
 ('telephone=none', 2.3974592252171386e-10),
 ('savings=... < 100 DM', 2.2430511967199792e-10),
 ('purpose=car (new)', -2.1264799070049253e-10),
 ('job=skilled employee / official', 2.115319882365941e-10),
 ('credit_history=existing credits

In [21]:
lime_exp.plot_features_importance()

alt.VConcatChart(...)

### LORE explainer

In [22]:
explainer = LoreTabularExplainer(bbox)
config = {'neigh_type':'geneticp', 'size':1000, 'ocr':0.1, 'ngen':10}
explainer.fit(df, class_field, config)
exp = explainer.explain(inst)
print(exp)

In [23]:
exp.plotRules()

In [24]:
exp.plotCounterfactualRules()

In [25]:
rules =exp.expDict['rule']['premise']

In [26]:
rules

[{'att': 'age', 'op': '<=', 'thr': 20.726173400878906, 'is_continuous': True},
 {'att': 'credit_amount',
  'op': '>',
  'thr': -439.6443485021591,
  'is_continuous': True},
 {'att': 'purpose=retraining',
  'op': '<=',
  'thr': 0.11524588242173195,
  'is_continuous': True},
 {'att': 'duration_in_month',
  'op': '>',
  'thr': -1.9407005310058594,
  'is_continuous': True},
 {'att': 'purpose=furniture/equipment',
  'op': '<=',
  'thr': 0.18370826542377472,
  'is_continuous': True},
 {'att': 'foreign_worker=no',
  'op': '<=',
  'thr': 0.7168410122394562,
  'is_continuous': True},
 {'att': 'purpose=domestic appliances',
  'op': '<=',
  'thr': 1.015466570854187,
  'is_continuous': True},
 {'att': 'savings=.. >= 1000 DM ',
  'op': '<=',
  'thr': 0.7176859378814697,
  'is_continuous': True},
 {'att': 'purpose=(vacation - does not exist?)',
  'op': '<=',
  'thr': 0.4622504562139511,
  'is_continuous': True},
 {'att': 'credit_history=critical account/ other credits existing (not at this bank)',
 

In [27]:
for r in rules:
    print(r['att'])

age
credit_amount
purpose=retraining
duration_in_month
purpose=furniture/equipment
foreign_worker=no
purpose=domestic appliances
savings=.. >= 1000 DM 
purpose=(vacation - does not exist?)
credit_history=critical account/ other credits existing (not at this bank)
people_under_maintenance


In [28]:
df_rules = pd.DataFrame.from_records(rules)

In [29]:
df.describe()

,duration_in_month,credit_amount,installment_as_income_perc,present_res_since,age,credits_this_bank,people_under_maintenance,account_check_status=0 <= ... < 200 DM,account_check_status=< 0 DM,account_check_status=>= 200 DM / salary assignments for at least 1 year,...,housing=rent,job=management/ self-employed/ highly qualified employee/ officer,job=skilled employee / official,job=unemployed/ unskilled - non-resident,job=unskilled - resident,telephone=none,"telephone=yes, registered under the customers name",foreign_worker=no,foreign_worker=yes,default
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,...,1000.000000,1000.000000,1000.000000,1000.000000,1000.0000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,20.903000,3271.258000,2.973000,2.845000,35.546000,1.407000,1.155000,0.269000,0.274000,0.063000,...,0.179000,0.148000,0.630000,0.022000,0.2000,0.596000,0.404000,0.037000,0.963000,0.300000
std,12.058814,2822.736876,1.118715,1.103718,11.375469,0.577654,0.362086,0.443662,0.446232,0.243085,...,0.383544,0.355278,0.483046,0.146757,0.4002,0.490943,0.490943,0.188856,0.188856,0.458487
min,4.000000,250.000000,1.000000,1.000000,19.000000,1.000000,1.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,12.000000,1365.500000,2.000000,2.000000,27.000000,1.000000,1.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,1.000000,0.000000
50%,18.000000,2319.500000,3.000000,3.000000,33.000000,1.000000,1.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.0000,1.000000,0.000000,0.000000,1.000000,0.000000
75%,24.000000,3972.250000,4.000000,4.000000,42.000000,2.000000,1.000000,1.000000,1.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.0000,1.000000,1.000000,0.000000,1.000000,1.000000
max,72.000000,18424.000000,4.000000,4.000000,75.000000,4.000000,2.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.0000,1.000000,1.000000,1.000000,1.000000,1.000000


In [30]:
df_range=pd.concat(
    {'min':X_train.min(),
     'max':X_train.max(),
     'std':X_train.std(),
     'q1':X_train.quantile(0.25),
     'median':X_train.quantile(0.50),
     'q3':X_train.quantile(0.75),
     },axis=1)

In [31]:
df_range=df_range.reset_index()

In [32]:
df_range

,index,min,max,std,q1,median,q3
0,duration_in_month,4,60,11.962777,12.00,18.0,24.00
1,credit_amount,338,15945,2674.942042,1360.75,2319.5,3974.50
2,installment_as_income_perc,1,4,1.123528,2.00,3.0,4.00
3,present_res_since,1,4,1.097089,2.00,3.0,4.00
4,age,19,75,10.954080,27.00,33.0,41.25
...,...,...,...,...,...,...,...
56,job=unskilled - resident,0,1,0.400286,0.00,0.0,0.00
57,telephone=none,0,1,0.491104,0.00,1.0,1.00
58,"telephone=yes, registered under the customers ...",0,1,0.491104,0.00,0.0,1.00
59,foreign_worker=no,0,1,0.196099,0.00,0.0,0.00


In [33]:
df_viz = df_range.merge(df_rules,how='left',left_on='index',right_on='att')
df_viz = df_viz.drop('att', axis=1)
df_viz

,index,min,max,std,q1,median,q3,op,thr,is_continuous
0,duration_in_month,4,60,11.962777,12.00,18.0,24.00,>,-1.940701,True
1,credit_amount,338,15945,2674.942042,1360.75,2319.5,3974.50,>,-439.644349,True
2,installment_as_income_perc,1,4,1.123528,2.00,3.0,4.00,NaN,NaN,NaN
3,present_res_since,1,4,1.097089,2.00,3.0,4.00,NaN,NaN,NaN
4,age,19,75,10.954080,27.00,33.0,41.25,<=,20.726173,True
...,...,...,...,...,...,...,...,...,...,...
56,job=unskilled - resident,0,1,0.400286,0.00,0.0,0.00,NaN,NaN,NaN
57,telephone=none,0,1,0.491104,0.00,1.0,1.00,NaN,NaN,NaN
58,"telephone=yes, registered under the customers ...",0,1,0.491104,0.00,0.0,1.00,NaN,NaN,NaN
59,foreign_worker=no,0,1,0.196099,0.00,0.0,0.00,<=,0.716841,True


In [34]:
thr2_list=[]
for i, row in df_viz.iterrows():
    if row['op']== '>' or row['op']== '>=':
        thr2_list.append(row['max'])
        continue
    if row['op']== '<' or row['op']== '<=':
        thr2_list.append(row['min'])
        continue
    else:
        thr2_list.append(np.nan)
df_viz['thr2'] = thr2_list
df_viz

,index,min,max,std,q1,median,q3,op,thr,is_continuous,thr2
0,duration_in_month,4,60,11.962777,12.00,18.0,24.00,>,-1.940701,True,60.0
1,credit_amount,338,15945,2674.942042,1360.75,2319.5,3974.50,>,-439.644349,True,15945.0
2,installment_as_income_perc,1,4,1.123528,2.00,3.0,4.00,NaN,NaN,NaN,NaN
3,present_res_since,1,4,1.097089,2.00,3.0,4.00,NaN,NaN,NaN,NaN
4,age,19,75,10.954080,27.00,33.0,41.25,<=,20.726173,True,19.0
...,...,...,...,...,...,...,...,...,...,...,...
56,job=unskilled - resident,0,1,0.400286,0.00,0.0,0.00,NaN,NaN,NaN,NaN
57,telephone=none,0,1,0.491104,0.00,1.0,1.00,NaN,NaN,NaN,NaN
58,"telephone=yes, registered under the customers ...",0,1,0.491104,0.00,0.0,1.00,NaN,NaN,NaN,NaN
59,foreign_worker=no,0,1,0.196099,0.00,0.0,0.00,<=,0.716841,True,0.0


In [35]:
for string, value in lime_feature_imp:
    print(value)
    break

-1.3883374336762273e-09


In [36]:
def add_fi_to_df_and_sort(dataframe, values):
    for string, value in values:
        dataframe.loc[dataframe['index'] == string, 'feature_importance'] = value
        dataframe.sort_values(by=['feature_importance'], key=lambda x: abs(x), ascending=False, inplace=True)
    return dataframe
df_fi=add_fi_to_df_and_sort(df_viz,lime_feature_imp)
df_fi

,index,min,max,std,q1,median,q3,op,thr,is_continuous,thr2,feature_importance
40,other_debtors=co-applicant,0,1,0.202680,0.0,0.0,0.0,NaN,NaN,NaN,NaN,-1.388337e-09
11,credit_history=all credits at this bank paid b...,0,1,0.226743,0.0,0.0,0.0,NaN,NaN,NaN,NaN,-1.122412e-09
35,present_emp_since=unemployed,0,1,0.223908,0.0,0.0,0.0,NaN,NaN,NaN,NaN,-1.032911e-09
42,other_debtors=none,0,1,0.288424,1.0,1.0,1.0,NaN,NaN,NaN,NaN,8.172969e-10
50,housing=for free,0,1,0.309516,0.0,0.0,0.0,NaN,NaN,NaN,NaN,-5.164340e-10
...,...,...,...,...,...,...,...,...,...,...,...,...
0,duration_in_month,4,60,11.962777,12.0,18.0,24.0,>,-1.940701,True,60.0,5.589211e-11
22,purpose=furniture/equipment,0,1,0.092250,0.0,0.0,0.0,<=,0.183708,True,0.0,4.222820e-11
48,other_installment_plans=none,0,1,0.397033,1.0,1.0,1.0,NaN,NaN,NaN,NaN,4.032642e-11
25,purpose=retraining,0,1,0.084273,0.0,0.0,0.0,<=,0.115246,True,0.0,3.525525e-11


In [37]:
df_viz['inst'] = real_inst.tolist()
df_viz

,index,min,max,std,q1,median,q3,op,thr,is_continuous,thr2,feature_importance,inst
40,other_debtors=co-applicant,0,1,0.202680,0.0,0.0,0.0,NaN,NaN,NaN,NaN,-1.388337e-09,27
11,credit_history=all credits at this bank paid b...,0,1,0.226743,0.0,0.0,0.0,NaN,NaN,NaN,NaN,-1.122412e-09,4526
35,present_emp_since=unemployed,0,1,0.223908,0.0,0.0,0.0,NaN,NaN,NaN,NaN,-1.032911e-09,4
42,other_debtors=none,0,1,0.288424,1.0,1.0,1.0,NaN,NaN,NaN,NaN,8.172969e-10,2
50,housing=for free,0,1,0.309516,0.0,0.0,0.0,NaN,NaN,NaN,NaN,-5.164340e-10,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,duration_in_month,4,60,11.962777,12.0,18.0,24.0,>,-1.940701,True,60.0,5.589211e-11,1
22,purpose=furniture/equipment,0,1,0.092250,0.0,0.0,0.0,<=,0.183708,True,0.0,4.222820e-11,0
48,other_installment_plans=none,0,1,0.397033,1.0,1.0,1.0,NaN,NaN,NaN,NaN,4.032642e-11,1
25,purpose=retraining,0,1,0.084273,0.0,0.0,0.0,<=,0.115246,True,0.0,3.525525e-11,0


In [38]:
features=df_viz['index'].to_list()
features

['other_debtors=co-applicant',
 'credit_history=all credits at this bank paid back duly',
 'present_emp_since=unemployed',
 'other_debtors=none',
 'housing=for free',
 'job=management/ self-employed/ highly qualified employee/ officer',
 'property=unknown / no property',
 'credit_amount',
 'housing=own',
 'savings=unknown/ no savings account',
 'people_under_maintenance',
 'foreign_worker=yes',
 'credits_this_bank',
 'telephone=none',
 'savings=... < 100 DM',
 'purpose=car (new)',
 'job=skilled employee / official',
 'credit_history=existing credits paid back duly till now',
 'account_check_status=0 <= ... < 200 DM',
 'age',
 'property=if not A121/A122 : car or other, not in attribute 6',
 'present_emp_since=1 <= ... < 4 years',
 'housing=rent',
 'purpose=(vacation - does not exist?)',
 'job=unskilled - resident',
 'present_emp_since=.. >= 7 years',
 'property=real estate',
 'property=if not A121 : building society savings agreement/ life insurance',
 'account_check_status=no checking 

In [39]:
df_rules = pd.DataFrame.from_records(rules)
df_rules

,att,op,thr,is_continuous
0,age,<=,20.726173,True
1,credit_amount,>,-439.644349,True
2,purpose=retraining,<=,0.115246,True
3,duration_in_month,>,-1.940701,True
4,purpose=furniture/equipment,<=,0.183708,True
5,foreign_worker=no,<=,0.716841,True
6,purpose=domestic appliances,<=,1.015467,True
7,savings=.. >= 1000 DM,<=,0.717686,True
8,purpose=(vacation - does not exist?),<=,0.462250,True
9,credit_history=critical account/ other credits...,<=,0.908596,True


In [40]:
def single_rule_plot_n(dataframe, rw):
    p=alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).mark_point(
        color='black' if rw['is_continuous'] == True else 'black',
        size=20,
        shape='diamond'
    ).encode(
        x=alt.X(
            field='inst',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        tooltip=[alt.Tooltip(field='inst', title=rw['index'])]
    )

    t_min = alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).mark_text(
        color='black',
        dx=-10,
        align='right'
    ).encode(
        x=alt.X(
            field='min',
            type='quantitative',
            title=None
        ),
        text='min:N'
    )

    t_max = alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).mark_text(
        color='black',
        dx=10,
        align='left'
    ).encode(
        x=alt.X(
            field='max',
            type='quantitative',
            title=None
        ),
        text='max:N'
    )

    q1_m = alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).mark_bar(
        color='#DAE7E8',
        size=12
    ).encode(
        x=alt.X(
            field='q1',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        x2 = alt.X2(
            field='median'
        ),
    )

    m_q3 = alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).mark_bar(
        color='#A8B9BF',
        size=12
    ).encode(
        x=alt.X(
            field='median',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        x2 = alt.X2(
            field='q3'
        ),
    )

    b =alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).mark_bar(
        color='#f28e46',size=5
    ).encode(
        x=alt.X(
            field='thr',
            type='quantitative',
            title=None,
        ),
        x2='thr2',
        y=alt.Y(
            field='index',
            type='nominal',
            title=None
        ),

    )



    l =alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).mark_bar(
        color='grey',size=1
    ).encode(
        x=alt.X(
            field='min',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        x2='max',
        y=alt.Y(field='index',type='nominal',title=None, axis=alt.Axis(labels= False, ticks=False))
    )


    return alt.layer(l,q1_m,m_q3,b,t_min,t_max,p).properties(
        height=12,
        width=399,
    )

In [41]:
def single_feature_importance_plot(dataframe, rw):

    chart = alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).mark_bar(
    ).encode(
        x=alt.X(
            field='feature_importance',
            type='quantitative',
            title=None,
            scale=alt.Scale(
                domain=(dataframe['feature_importance'].min(), dataframe['feature_importance'].max()),
                nice=False
            )
        ),
        y=alt.Y(
            field='index',
            type='nominal',
            title=None,
            axis=None
        ),
        color=alt.condition('datum.feature_importance > 0', alt.value('#2C0AD1'), alt.value('#DB2C8F') ),
    )
    return chart.properties(
        height=12,
        width=50
    )

In [42]:
def single_instance_text(dataframe, rw):
    chart = alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).mark_text(
        color='black',
        align='left',
        dx=-30,
        fontWeight='bold'
    ).encode(
            text=alt.Text(
            field='inst',
            type='quantitative',
            title=None
        )
    )
    return chart.properties(
        height=12,
        width=10
    )

In [43]:
def single_index_text(dataframe, rw):
    chart = alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).transform_calculate(
        label ="datum.type=='categorical' ? datum.index : datum.index +' = '+ datum.inst" #  datum.index +' = '+ datum.inst
    ).mark_text(
        color='black',
        align='left',
        dx=-100,
    ).encode(
            text=alt.Text(
            field='label',
            type='nominal',
            title=None
        )
    )
    return chart.properties(
        height=12,
        width=50
    )

In [44]:
def single_rule_plot_q(dataframe, rw):
    name= rw['index'].split('=')[0]
    data = dataframe[dataframe['rname'] == name]
    b = alt.Chart(
        data
    ).mark_bar(
        stroke='white'
    ).encode(
        x=alt.X(
            field='count',
            type='quantitative',
            title=None,
            # stack="normalize"
        ),
        y=alt.Y(
            field='rname',
            type='nominal',
            axis=None
        ),
        detail='index:N',
        color=alt.condition('datum.inst==1',alt.value('darkgrey'),alt.value('lightgrey'))
    )

    r=alt.Chart(
        data
    ).mark_bar(
        stroke='white'
    ).encode(
        x=alt.X(
            field='count',
            type='quantitative',
            title=None,
            # stack="normalize"
        ),
        y=alt.Y(
            field='rname',
            type='nominal',
            axis=None
        ),
        detail='index:N',
        color=alt.condition('datum.is_continuous',alt.value('#f28e46'),alt.value('white')),
        opacity=alt.condition('datum.is_continuous',alt.value(1),alt.value(0.001)),
        tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
    )
    
    dot =alt.Chart(
        data
    ).transform_stack(
        stack='count',
        as_=['count_start','count_end'],
        groupby=['rname'],
        sort=[alt.SortField('count', 'descending')]
    ).mark_point(
        size=40,
        shape='diamond',
        color='black'
    ).encode(
        x=alt.X(
            field='count_start',
            type='quantitative',
            title=None,
            # stack="normalize",
        ),
        x2=alt.X2(
            field='count_end',
            type='quantitative',
            title=None,
            # stack="normalize",
        ),
        y=alt.Y(
            field='rname',
            type='nominal',
            axis=None
        ),
        detail='index:N',
        # color=alt.condition('datum.is_continuous',alt.value('#f28e46'),alt.value('white')),
        # opacity=alt.condition('datum.is_continuous',alt.value(1),alt.value(0.001)),
        #tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
    )
    

    return alt.layer(b,r).properties(
        height=10,
        width=400,
    )

In [45]:
def plot_rules(dataframe):
    tx_list=[]
    ti_list=[]
    rp_list=[]
    fi_list=[]
    for i, row in dataframe.iterrows():
        if row['inst']!=0: # or row['is_continuous']==True
            stx = single_instance_text(dataframe, row)
            sti = single_index_text(dataframe, row)
            if row['type']== 'numeric':
                srp = single_rule_plot_n(dataframe, row)
            else:
                srp = single_rule_plot_q(dataframe, row)
            sfi = single_feature_importance_plot(dataframe, row)
            tx_list.append(stx)
            ti_list.append(sti)
            rp_list.append(srp)
            fi_list.append(sfi)
    tx_concat=alt.vconcat(*tx_list)
    ti_concat=alt.vconcat(*ti_list)
    rp_concat=alt.vconcat(*rp_list, title='Rule')
    fi_concat=alt.vconcat(*fi_list, title='FI')
    final_chart = alt.hconcat(fi_concat, rp_concat, ti_concat)

    return final_chart.configure_concat(
        spacing=3
    ).configure_axis(
        grid=False
    ).configure_view(
        strokeWidth=0,
        stroke='lightgray'
    ).configure_axisX(
        disable=True
    ).configure_axisY(
        domain=False,
        ticks=False
    ).configure_title(
        fontWeight='bold',
        anchor='middle',

    )

In [46]:
def dot_on_categories(dataframe,sort_by_rules=True):
    base =alt.Chart(
        dataframe
    ).transform_stack(
        stack='count',
        as_=['count_start','count_end'],
        groupby=['rname'],
        sort=[alt.SortField('count', 'ascending')]
    ).transform_calculate(
        midStack='(datum.count_start+datum.count_end)/2'
    )
    
    bar = base.mark_bar(stroke='white').encode(
        x='count_start:Q',
        x2='count_end:Q',
        y=alt.Y('rname:N',sort=["is_continuous", "feature_importance"]),
        detail='index:N',
        color=alt.condition('datum.is_continuous',alt.value("#f4dd4d"),alt.value('lightgrey')),
        tooltip=[alt.Tooltip('category')]
    )
    
    dot= base.mark_point(
        color='black',
        shape='diamond',
        fill='black',
        size=30
    ).encode(
        x='midStack:Q',
        y=alt.Y('rname:N',sort=["is_continuous", "feature_importance"]),
        detail='index:N',
        opacity=alt.condition('datum.inst==1',alt.value(0.6),alt.value(0))
    )

    return (bar+dot).properties(
        width=200,
    ).configure_axisX(
        disable=True
    ).configure_axisY(
        labelPadding=10,
        labelFontSize=12,
        domain=False,
        ticks=False,
        title=None
    ).configure_axis(
        grid=True
    ).configure_view(
        strokeWidth=0,
        stroke='lightgray'
    )
dot_on_categories(df_v)

NameError: name 'df_v' is not defined

# Function to prepare data for plotting

In [47]:
def data_to_plot(
        feature_names=feature_names, real_feature_names=real_feature_names,
        instance_number=None, x_train=None, rules=None,
        feature_importance=None):
    feature_list =[]
    #Convert the list of tuples generated by lime in a dict
    if feature_importance is 'lime':
        lime_dict = {}
        for (key, value) in lime_feature_importance:
            lime_dict.setdefault(key, value)
    for i, el in enumerate(feature_names):
        f ={}
        if el in numeric_columns:
            f['type'] = 'numeric'
            f['name'] = el
            f['rname'] = real_feature_names[i]
            if x_train is not None:
                f['min'] = x_train[el].min()
                f['max'] = x_train[el].max()
                f['q1'] = x_train[el].quantile(0.25)
                f['median'] = x_train[el].quantile(0.50)
                f['q3'] = x_train[el].quantile(0.75)
                f['mean'] = x_train[el].mean()
                f['std'] = x_train[el].std()
        else:
            f['type'] = 'categorical'
            f['name'] = el
            f['rname'] = el.split('=')[0]
            f['category'] = el.split('=',1)[1]
            if x_train is not None:
                f['count'] = x_train[el].sum()

        if feature_importance is 'lime':
            f['feature_importance'] = lime_dict[el]
        if feature_importance is 'shap':
            f['feature_importance'] = shap_feature_importance[1][i]
        feature_list.append(f)
    df =pd.DataFrame.from_records(feature_list)

    if instance_number:
        inst = X_train.iloc[instance_number].values
        df['inst'] = inst
    if rules is not None:
        df_rules = pd.DataFrame.from_records(rules)
        df = df.merge(df_rules,how='left',left_on='name',right_on='att')
        df = df.drop('att', axis=1)
        thr2_list=[]
        for i, row in df.iterrows():
            if row['op']== '>' or row['op']== '>=':
                thr2_list.append(row['max'])
                continue
            if row['op']== '<' or row['op']== '<=':
                thr2_list.append(row['min'])
                continue
            else:
                thr2_list.append(np.nan)
        df['thr2'] = thr2_list
    df.sort_values(by=['feature_importance'], key=lambda x: abs(x), ascending=False, inplace=True)
    return df

In [48]:
df_v = data_to_plot(feature_names=feature_names, real_feature_names=real_feature_names, instance_number=3, x_train=X_train, rules=rules, shap_feature_importance=shap_feature_importance)
df_v

TypeError: data_to_plot() got an unexpected keyword argument 'shap_feature_importance'

In [49]:
plot_rules(df_v.rename(columns={'name':'index'}))

NameError: name 'df_v' is not defined

# DATI NARET

In [50]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [51]:
pip install dill

Note: you may need to restart the kernel to use updated packages.


In [52]:
pip install category-encoders

Note: you may need to restart the kernel to use updated packages.


In [53]:
import xgboost as xgb
import dill
import SuperLore
import category_encoders

### TRAIN MODEL

In [54]:
bb = xgb.XGBClassifier()
bb.load_model("../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_train.model")


In [55]:
bb

XGBClassifier(base_score=None, booster='gbtree', callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, early_stopping_rounds=50,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              gamma=0.65, gpu_id=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.025, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=13, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              n_estimators=110, n_jobs=10, num_parallel_tree=None,
              objective='binary:hinge', predictor=None, ...)

### X_train

In [56]:
X_train = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xtrain')
X_train

,PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO,PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO,PRODV_LETTERE_DI_CREDITO_PON,SCADV_FLG_RATA_DIVISA_SEK,OWNER_PRODV_FOREX_PON,OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON,PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM,PN_LC_IMPORT_FLG_ONLY_TY,PN_SEPA_ENTRATA_TY_VAL,SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE,...,PCRIV_FT_20_DLT_ANNUO_UTILZZ_ACCRD_MEDIO,SCADV_FLG_RATA_DIVISA_JPY,PN_SEPA_USCITA_TY_NUM,PRODV_FACTORING_PON,PRODV_SECURITIZATION_PON,BESTV_BON_VERSO_NON_EU_TY_VAL,PCRIV_MERCATO_EUR_TY,PCRIV_NAT_FT_03_LAST_YYYYMM_UTILZZ_ACCRDT_PERC,PCRIV_NAT_FT_07_LAST_YYYYMM_ACCORDATO,PN_SWIFT_USCITA_PRC_DLT_YEAR_VAL
0,0.190192,0.171171,0.452953,0.0,0.0,0.0,0.899817,0.0,0.969766,0.341267,...,0.148956,0.0,0.954954,0.462462,0.498498,0.727661,1.0,0.000000,0.0,0.568569
1,0.473459,0.635858,0.452953,0.0,0.0,0.0,0.212667,0.0,0.153186,0.028649,...,0.412056,0.0,0.136097,0.462462,0.498498,0.443647,1.0,0.000000,0.0,0.327034
2,0.561579,0.297557,0.452953,0.0,0.0,0.0,0.333834,0.0,0.283709,0.139380,...,0.554918,0.0,0.492492,0.462462,0.498498,0.323121,1.0,0.071071,0.0,0.195570
3,0.408324,0.495087,0.452953,0.0,0.0,0.0,0.184685,0.0,0.321320,0.431901,...,0.545948,0.0,0.158829,0.462462,0.498498,0.428852,1.0,0.639640,0.0,0.295616
4,0.321600,0.476636,0.452953,0.0,0.0,0.0,0.165893,0.0,0.030564,0.056967,...,0.479533,0.0,0.124761,0.462462,0.498498,0.426885,1.0,0.284284,0.0,0.270870
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4153,0.508898,0.086176,0.452953,0.0,0.0,0.0,0.583083,0.0,0.439938,0.240102,...,0.596489,0.0,0.752652,0.462462,0.498498,0.429894,1.0,0.000000,0.0,0.273627
4154,0.000000,0.409412,0.452953,0.0,0.0,0.0,0.282646,0.0,0.294971,0.015265,...,0.049131,0.0,0.051552,0.948644,0.498498,0.328031,1.0,0.000000,0.0,0.388330
4155,0.360037,0.533749,0.452953,0.0,0.0,0.0,0.092001,0.0,0.313670,0.127011,...,0.444565,0.0,0.147056,0.462462,0.498498,0.357598,1.0,0.000000,0.0,0.308467
4156,0.646133,0.221197,0.452953,0.0,0.0,0.0,0.701201,0.0,0.588533,0.167629,...,0.564699,0.0,0.607047,0.462462,0.498498,0.599600,1.0,0.000000,0.0,0.949939


### Y_train

In [57]:
Y_train = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_ytrain')
Y_train

1410    1
1737    0
2102    0
4745    0
5354    0
       ..
5095    0
3977    0
3322    0
5173    1
4773    0
Name: OWNER_PRODV_INCASSI_E_PAGAMENTI_INTERNAZIONALI_PON, Length: 4158, dtype: int64

### X_test

In [58]:
X_test = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xtest')
X_test

,PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO,PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO,PRODV_LETTERE_DI_CREDITO_PON,SCADV_FLG_RATA_DIVISA_SEK,OWNER_PRODV_FOREX_PON,OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON,PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM,PN_LC_IMPORT_FLG_ONLY_TY,PN_SEPA_ENTRATA_TY_VAL,SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE,...,PCRIV_FT_20_DLT_ANNUO_UTILZZ_ACCRD_MEDIO,SCADV_FLG_RATA_DIVISA_JPY,PN_SEPA_USCITA_TY_NUM,PRODV_FACTORING_PON,PRODV_SECURITIZATION_PON,BESTV_BON_VERSO_NON_EU_TY_VAL,PCRIV_MERCATO_EUR_TY,PCRIV_NAT_FT_03_LAST_YYYYMM_UTILZZ_ACCRDT_PERC,PCRIV_NAT_FT_07_LAST_YYYYMM_ACCORDATO,PN_SWIFT_USCITA_PRC_DLT_YEAR_VAL
0,0.000000,0.227975,0.990218,0.0,1.0,1.0,0.947305,0.0,0.921458,0.511439,...,0.306045,0.0,0.976949,0.462462,0.498498,0.933922,1.0,0.781782,0.0,0.534532
1,0.529808,0.324960,0.452953,0.0,0.0,0.0,0.758759,0.0,0.525933,0.239234,...,0.580794,0.0,0.619747,0.462462,0.498498,0.296540,1.0,0.071071,0.0,0.658905
2,0.523023,0.204011,0.452953,0.0,0.0,0.0,0.233552,0.0,0.026026,0.388234,...,0.022030,0.0,0.141733,0.923689,0.498498,0.386674,1.0,0.781782,0.0,0.141576
3,0.611950,0.291709,0.452953,0.0,0.0,0.0,0.130130,0.0,0.319746,0.311553,...,0.596059,0.0,0.338338,0.462462,0.498498,0.140292,1.0,0.355355,0.0,0.400205
4,0.556724,0.935637,0.452953,0.0,0.0,0.0,0.357976,0.0,0.341334,0.821015,...,0.479934,0.0,0.258342,0.462462,0.498498,0.435522,1.0,0.071071,0.0,0.314382
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0.400207,0.079736,0.452953,0.0,0.0,0.0,0.804805,0.0,0.511450,0.253987,...,0.512023,0.0,0.831394,0.462462,0.498498,0.250648,1.0,0.781782,0.0,0.491389
887,0.927949,0.423385,0.452953,0.0,0.0,0.0,0.980164,0.0,0.896275,0.412989,...,0.525526,0.0,0.985046,0.951338,0.498498,0.163370,1.0,0.999157,0.0,0.177406
888,0.565507,0.355052,0.452953,0.0,0.0,0.0,0.000000,0.0,0.278770,0.287375,...,0.589377,0.0,0.398398,0.462462,0.498498,0.261080,1.0,0.000000,0.0,0.381746
889,0.524776,0.296970,0.452953,0.0,0.0,0.0,0.000000,0.0,0.118861,0.220148,...,0.550981,0.0,0.276777,0.462462,0.498498,0.443712,1.0,0.781782,0.0,0.382447


### Y_test

In [59]:
Y_test = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_ytest')
Y_test

1943    1
1146    1
1493    0
395     1
4317    0
       ..
34      0
2016    1
4461    0
282     0
4706    0
Name: OWNER_PRODV_INCASSI_E_PAGAMENTI_INTERNAZIONALI_PON, Length: 891, dtype: int64

In [60]:
Y_test

1943    1
1146    1
1493    0
395     1
4317    0
       ..
34      0
2016    1
4461    0
282     0
4706    0
Name: OWNER_PRODV_INCASSI_E_PAGAMENTI_INTERNAZIONALI_PON, Length: 891, dtype: int64

### Data description

In [61]:
data_desc=pd.read_pickle('../datasets/Dati-Banca-Lore/intesa_incassi_data_description.p')
data_desc

{'numerical': {'PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO': {'mean': 0.4963,
   'std': 0.1677,
   'min': 0.0,
   'max': 1.0,
   '1st-quantile': 0.4274,
   '2nd-quantile': 0.5059,
   '3rd-quantile': 0.5592},
  'PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO': {'mean': 0.4103,
   'std': 0.2236,
   'min': 0.0,
   'max': 1.0,
   '1st-quantile': 0.246,
   '2nd-quantile': 0.3661,
   '3rd-quantile': 0.5307},
  'PRODV_LETTERE_DI_CREDITO_PON': {'mean': 0.5002,
   'std': 0.1472,
   'min': 0.0,
   'max': 1.0,
   '1st-quantile': 0.453,
   '2nd-quantile': 0.453,
   '3rd-quantile': 0.453},
  'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM': {'mean': 0.4061,
   'std': 0.2854,
   'min': 0.0,
   'max': 1.0,
   '1st-quantile': 0.1745,
   '2nd-quantile': 0.3338,
   '3rd-quantile': 0.6341},
  'PN_SEPA_ENTRATA_TY_VAL': {'mean': 0.4587,
   'std': 0.2845,
   'min': 0.0,
   'max': 1.0,
   '1st-quantile': 0.2157,
   '2nd-quantile': 0.4202,
   '3rd-quantile': 0.7006},
  'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE': {'mean': 0.3707,
   's

### Lore exp

In [62]:
lore_exp_path=open('../datasets/Dati-Banca-Lore/lore_exp_INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xgb_cfs_binary_from_dts.p','rb')

In [66]:
objs = []
while 1:
    try:
        objs.append(pickle.load(lore_exp_path))
    except EOFError:
        break

In [67]:
objs

In [ ]:
print(objs[0])

for c in X_train.columns:
        if X_train[c].max() == 1.0 and X_train[c].min() == 0.0:
            print('Colonna categorica! ', c)
        else:
            numeric_columns.append(c)

In [82]:
prova = objs[0].rule.class_name
prova

'Target'

class Rule(object):

    def __init__(self, premises, cons, class_name):
        self.premises = premises
        self.cons = cons
        self.class_name = class_name

In [86]:
for p in prova:
    print(p.att)
    print(p.op)

AttributeError: 'str' object has no attribute 'att'

In [75]:
print(objs[1])

r = { PN_SEPA_USCITA_TY_NUM = 1.00, PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM = 0.00, SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE = 0.00 } --> { Target: 1 }
c = { {  } }
fi = {'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM': 0.08715315227966847, 'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE': 0.15527019210190482, 'PN_SEPA_USCITA_TY_NUM': 0.41146357278564144}
fia = {'PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO': 0.04178030953650566, 'PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO': 0.0, 'PRODV_LETTERE_DI_CREDITO_PON': 0.0, 'SCADV_FLG_RATA_DIVISA_SEK': 0.0, 'OWNER_PRODV_FOREX_PON': 0.15758687190360676, 'OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON': 0.0, 'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM': 0.08715315227966847, 'PN_LC_IMPORT_FLG_ONLY_TY': 0.0, 'PN_SEPA_ENTRATA_TY_VAL': 0.0, 'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE': 0.15527019210190482, 'OWNER_PRODV_GARANZIE_DOMESTICHE_PON': 0.0, 'PN_SEPA_USCITA_PRC_DLT_YEAR_VAL': 0.0, 'SCADV_FLG_RATA_DIVISA_AUD': 8.884635324177429e-17, 'OWNER_PRODV_ANTICIPI_EXPORT_PON': 0.0, 'PN_SWIFT_ENTRATA_FLG_NEW_TRA

In [78]:
for i in objs[3].crules:
    print(i)

{ PN_SWIFT_USCITA_PRC_DLT_YEAR_VAL = 1.00, OWNER_PRODV_ANTICIPI_EXPORT_PON = 0.00, OWNER_PRODV_FOREX_PON = 1.00, PN_LC_IMPORT_FLG_ONLY_TY = 0.00 } --> { Target: 1 }


In [ ]:
rules = exp.expDict['rule']['premise']

In [ ]:
print(rules[1].exemplars)

In [ ]:
rules =rules['rule']['premise']

In [ ]:
# check sulle categoriche, cambiare nella colonna type

### SHAP FI FULL

In [87]:
#carico shapley value full
path =('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_explanations_full.p')
explanations_shap = dill.load(open(path, 'rb'))

In [88]:
explanations_shap # non è la stessa struttura dell'altro shap classico

array([[ 0.09466362,  0.08120622,  0.28542907, ..., -0.07626389,
         0.        , -0.02731891],
       [-0.02177902,  0.07350576, -0.01937944, ...,  0.0781945 ,
         0.        ,  0.96232297],
       [ 0.01969376,  0.08790127, -0.03470209, ..., -0.16089288,
         0.        , -0.26676439],
       ...,
       [ 0.02056669,  0.08696288, -0.03176322, ...,  0.13737915,
         0.        , -0.17642036],
       [-0.00200953,  0.06920489, -0.03739386, ..., -0.07819865,
         0.        , -0.19285882],
       [ 0.04856581, -0.03532057, -0.01439595, ..., -0.01933182,
         0.        ,  0.06414079]])

In [65]:
inst = X_train.iloc[1].values
print('Instance ',inst)
print('True class ',Y_train.iloc[128])
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

Instance  [0.47345931 0.63585823 0.45295295 0.         0.         0.
 0.21266721 0.         0.15318606 0.02864901 0.         0.5266462
 0.         0.         0.         0.         0.         0.41205649
 0.         0.13609707 0.46246246 0.4984985  0.44364682 1.
 0.         0.         0.32703369]
True class  1
Predicted class  [0]


### Create the df

In [83]:
feature_names = X_test.columns
real_feature_names = X_test.columns

In [84]:
feature_names

Index(['PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO',
       'PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO',
       'PRODV_LETTERE_DI_CREDITO_PON', 'SCADV_FLG_RATA_DIVISA_SEK',
       'OWNER_PRODV_FOREX_PON', 'OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON',
       'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM', 'PN_LC_IMPORT_FLG_ONLY_TY',
       'PN_SEPA_ENTRATA_TY_VAL', 'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE',
       'OWNER_PRODV_GARANZIE_DOMESTICHE_PON',
       'PN_SEPA_USCITA_PRC_DLT_YEAR_VAL', 'SCADV_FLG_RATA_DIVISA_AUD',
       'OWNER_PRODV_ANTICIPI_EXPORT_PON', 'PN_SWIFT_ENTRATA_FLG_NEW_TRANS',
       'SCADV_FLG_RATA_DIVISA_CAD', 'OWNER_PRODV_DCM_PON',
       'PCRIV_FT_20_DLT_ANNUO_UTILZZ_ACCRD_MEDIO', 'SCADV_FLG_RATA_DIVISA_JPY',
       'PN_SEPA_USCITA_TY_NUM', 'PRODV_FACTORING_PON',
       'PRODV_SECURITIZATION_PON', 'BESTV_BON_VERSO_NON_EU_TY_VAL',
       'PCRIV_MERCATO_EUR_TY',
       'PCRIV_NAT_FT_03_LAST_YYYYMM_UTILZZ_ACCRDT_PERC',
       'PCRIV_NAT_FT_07_LAST_YYYYMM_ACCORDATO',
       'PN_SWIFT_USCITA

In [104]:
temp={}
i=0
df_viz = pd.DataFrame(columns = ['type', 'name', 'rname', 'min', 'max', 'q1', 'median', 'q3', 'mean',
       'std', 'feature_importance', 'category', 'count', 'inst', 'op', 'thr',
       'is_continuous', 'thr2'])
i_f=0
bool_f= False
for f in feature_names:
    temp[f]= dict()
    temp[f]['feature_importance'] = explanations_shap[i][i_f]
    for lore_p in objs[i].rule.premises:
        if lore_p.att == f:
            op = lore_p.op
            thr = lore_p.thr
            is_continuous = lore_p.is_continuous
            temp[f]['op']= op
            temp[f]['thr']= thr
            temp[f]['is_continuous']= is_continuous
            bool_f= True
    if len(np.unique(X_train[f]))==2:
        type_f = 'categorical'
        count=np.unique(X_train[f], return_counts= True)
        temp[f]['type']= type_f
        temp[f]['count']= count
    else:
        type_f = 'numeric'
        temp[f]['type']= type_f
        min_f = X_train[f].min()
        temp[f]['min']= min_f
        max_f = X_train[f].max()
        temp[f]['max']= max_f
        q_1 =X_train[f].quantile(0.25)
        temp[f]['q1']= q_1
        median =X_train[f].quantile(0.50)
        temp[f]['median']= median
        q_3=X_train[f].quantile(0.75)
        temp[f]['q3']= q_3
        temp[f]['mean'] = X_train[f].mean()
        temp[f]['std']=X_train[f].std()
        if bool_f == True:
            if op == '>=':
                temp[f]['thr_2'] = max_f
            else:
                temp[f]['thr_2'] = min_f
        bool_f=False
    i_f+=1

In [105]:
temp['PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO']

{'feature_importance': 0.09466361575103292,
 'type': 'numeric',
 'min': 0.0,
 'max': 1.0,
 'q1': 0.427429984585868,
 'median': 0.5059183031631036,
 'q3': 0.5592064079239055,
 'mean': 0.49634247694534256,
 'std': 0.16774322852151824}

In [106]:
temp

{'PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO': {'feature_importance': 0.09466361575103292,
  'type': 'numeric',
  'min': 0.0,
  'max': 1.0,
  'q1': 0.427429984585868,
  'median': 0.5059183031631036,
  'q3': 0.5592064079239055,
  'mean': 0.49634247694534256,
  'std': 0.16774322852151824},
 'PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO': {'feature_importance': 0.08120621911794842,
  'type': 'numeric',
  'min': 0.0,
  'max': 1.0,
  'q1': 0.24601064952760232,
  'median': 0.36613786850973884,
  'q3': 0.5307447645634663,
  'mean': 0.41028775969789066,
  'std': 0.22355765029927188},
 'PRODV_LETTERE_DI_CREDITO_PON': {'feature_importance': 0.2854290714347371,
  'type': 'numeric',
  'min': 0.0,
  'max': 1.0,
  'q1': 0.45295295295295296,
  'median': 0.45295295295295296,
  'q3': 0.45295295295295296,
  'mean': 0.5002006505582941,
  'std': 0.14716965737121998},
 'SCADV_FLG_RATA_DIVISA_SEK': {'feature_importance': 0.0,
  'op': '=',
  'thr': 1,
  'is_continuous': False,
  'type': 'numeric',
  'min': 0.0,
  'ma

In [118]:
df_viz = df_viz.from_dict(temp) 


In [119]:
df_viz = df_viz.T.reset_index()

In [120]:
df_viz=df_viz.rename(columns={'index':'name'})
df_viz['rname']=df_viz['name']

In [121]:
df_viz

,name,feature_importance,type,min,max,q1,median,q3,mean,std,op,thr,is_continuous,thr_2,count,rname
0,PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO,0.094664,numeric,0.0,1.0,0.42743,0.505918,0.559206,0.496342,0.167743,NaN,NaN,NaN,NaN,NaN,PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO
1,PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO,0.081206,numeric,0.0,1.0,0.246011,0.366138,0.530745,0.410288,0.223558,NaN,NaN,NaN,NaN,NaN,PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO
2,PRODV_LETTERE_DI_CREDITO_PON,0.285429,numeric,0.0,1.0,0.452953,0.452953,0.452953,0.500201,0.14717,NaN,NaN,NaN,NaN,NaN,PRODV_LETTERE_DI_CREDITO_PON
3,SCADV_FLG_RATA_DIVISA_SEK,0.0,numeric,0.0,0.0,0.0,0.0,0.0,0.0,0.0,=,1,False,0.0,NaN,SCADV_FLG_RATA_DIVISA_SEK
4,OWNER_PRODV_FOREX_PON,0.334717,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"([0.0, 1.0], [3800, 358])",OWNER_PRODV_FOREX_PON
5,OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON,0.115931,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"([0.0, 1.0], [3975, 183])",OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON
6,PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM,0.037198,numeric,0.0,1.0,0.174504,0.333834,0.634134,0.406126,0.285385,NaN,NaN,NaN,NaN,NaN,PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM
7,PN_LC_IMPORT_FLG_ONLY_TY,0.0,numeric,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,PN_LC_IMPORT_FLG_ONLY_TY
8,PN_SEPA_ENTRATA_TY_VAL,0.182715,numeric,0.0,1.0,0.215689,0.420236,0.700645,0.45874,0.284519,NaN,NaN,NaN,NaN,NaN,PN_SEPA_ENTRATA_TY_VAL
9,SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE,-0.024854,numeric,0.0,1.0,0.25924,0.331998,0.431406,0.370697,0.17931,NaN,NaN,NaN,NaN,NaN,SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE
